## Импорты

In [3]:
import sys
from pathlib import Path

current = Path.cwd()
project_root = None
for p in [current] + list(current.parents):
    if (p / "src").is_dir() and (p / "requirements.txt").is_file():
        project_root = p
        break

if project_root is None:
    project_root = current

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import logging
import os
from typing import List, Optional
from dotenv import load_dotenv

import kaggle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_curve, auc

from src.models.preprocessor import DataPreprocessor
from src.models.evaluator import ModelEvaluator
from src.models.classifier import FitnessClassifier


In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)
logger = logging.getLogger(__name__)

In [6]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10

## Предобработка данных

In [8]:
def preprocess_real_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Адаптирует сырой датасет под ожидаемую схему."""
    df = df.copy()
    
    if "smokes" in df.columns:
        df["smokes"] = df["smokes"].astype(str).str.lower().map(
            lambda x: 1 if x in ["1", "yes", "true"] else 0
        )
    
    if "is_fit" in df.columns:
        df["is_fit"] = pd.to_numeric(df["is_fit"], errors="coerce").fillna(0).astype(int)
        df = df.dropna(subset=["is_fit"])
    
    logger.info(f"Preprocessed: {len(df)} rows, columns: {list(df.columns)}")
    return df

## Загрузка датасета

In [ ]:
def load_kaggle_credentials() -> tuple[str, str]:
    """Загружает Kaggle credentials из configs/.env или окружения."""
    env_path = project_root / "configs" / ".env"
    
    if env_path.exists():
        try:
            load_dotenv(env_path)
            logger.info(f"Loaded credentials from {env_path}")
        except ImportError:
            logger.warning("python-dotenv not installed")
    
    username = os.getenv("KAGGLE_USERNAME", "").strip().strip('"').strip("'")
    key = os.getenv("KAGGLE_KEY", "").strip().strip('"').strip("'")
    
    if not username or not key:
        raise RuntimeError(
            f"Kaggle credentials not found. Set in {project_root / 'configs' / '.env'}"
        )
    return username, key


def download_kaggle_dataset(
    dataset_slug: str,
    download_dir: str = "data",
    target_filename: str = "fitness_dataset.csv"
) -> Path:
    """Скачивает датасет с Kaggle."""
    
    download_path = Path(download_dir)
    download_path.mkdir(parents=True, exist_ok=True)
    
    username, key = load_kaggle_credentials()
    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = key
    
    logger.info(f"Downloading '{dataset_slug}' to {download_path}...")
    
    try:
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            dataset_slug,
            path=str(download_path),
            unzip=True,
            quiet=False
        )
        logger.info("Kaggle API download complete")
    except Exception as e:
        logger.error(f"Kaggle API failed: {e}")
        raise RuntimeError(f"Failed to download dataset via Kaggle API: {e}")
    
    csv_files = list(download_path.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {download_path}")
    
    target = None
    for f in csv_files:
        if target_filename.lower() in f.name.lower():
            target = f
            break
    if target is None:
        target = csv_files[0]
        
    logger.info(f"Dataset downloaded: {target}")
    return target


def load_dataset(
    path: str = "data/fitness_dataset.csv",
    kaggle_slug: Optional[str] = None
) -> pd.DataFrame:
    """Загружает датасет."""
    data_path = project_root / path
    
    if data_path.exists():
        logger.info(f"Loading from {data_path}")
        return pd.read_csv(data_path)
    
    if kaggle_slug:
        logger.warning(f"File not found. Downloading from Kaggle: {kaggle_slug}")
        downloaded = download_kaggle_dataset(kaggle_slug, str(project_root / "data"), "fitness_dataset.csv")
        return pd.read_csv(downloaded)
    
    raise FileNotFoundError(
        f"Dataset not found at {data_path}. Place it there or set KAGGLE_SLUG."
    )

In [11]:
def run_eda(df: pd.DataFrame) -> None:
    """Разведочный анализ данных."""
    logger.info("EDA: Quick Checks")
    
    print(f"Dataset shape: {df.shape}")
    
    if "is_fit" in df.columns:
        print(f"Target distribution:\n{df['is_fit'].value_counts(normalize=True).apply(lambda x: f'{x:.2%}')}")
    
    print(f"First 5 rows:\n{df.head().to_string()}")
    print(f"Describe:\n{df.describe().round(2)}")
    
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"Missing values:\n{missing[missing > 0]}")
    
    numeric_cols = ["age", "height_cm", "weight_kg", "sleep_hours", "activity_index"]
    existing = [c for c in numeric_cols if c in df.columns]
    
    if "is_fit" in df.columns and existing:
        corr = df[existing + ["is_fit"]].corr()["is_fit"].drop("is_fit").sort_values(ascending=False)
        print(f"Correlation with 'is_fit':\n{corr.apply(lambda x: f'{x:+.3f}')}")

## Визуализация

In [ ]:
def save_plots(
    df: pd.DataFrame,
    y_test: np.ndarray,
    y_pred_base: np.ndarray,
    y_pred_rf: np.ndarray,
    y_proba_base: np.ndarray,
    y_proba_rf: np.ndarray,
    feature_names: List[str],
    importances: np.ndarray,
    output_dir: Path
) -> None:
    """
    Генерирует и сохраняет графики в папку artifacts/.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Saving plots to {output_dir}")
    
    # 1. Распределение целевой переменной
    plt.figure(figsize=(6, 4))
    if "is_fit" in df.columns:
        df["is_fit"].value_counts().plot(kind="bar", color=["#ff6b6b", "#4ecdc4"], edgecolor="black")
        plt.title("Distribution of Target Variable (is_fit)")
        plt.xlabel("is_fit")
        plt.ylabel("Count")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(output_dir / "01_target_distribution.png", dpi=300)
        plt.close()

    # 2. Тепловая карта корреляций
    numeric_cols = ["age", "height_cm", "weight_kg", "sleep_hours", "activity_index"]
    existing_num = [c for c in numeric_cols if c in df.columns]
    
    if existing_num and "is_fit" in df.columns:
        plt.figure(figsize=(10, 8))
        corr_matrix = df[existing_num + ["is_fit"]].corr()
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
        plt.title("Correlation Heatmap")
        plt.tight_layout()
        plt.savefig(output_dir / "02_correlation_heatmap.png", dpi=300)
        plt.close()
    
    # 3. Матрицы ошибок
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    cm_base = confusion_matrix(y_test, y_pred_base)
    sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues", ax=axes[0])
    axes[0].set_title("Baseline: Logistic Regression")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    
    cm_rf = confusion_matrix(y_test, y_pred_rf)
    sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[1])
    axes[1].set_title("Improved: Random Forest")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")
    
    plt.suptitle("Confusion Matrices Comparison")
    plt.tight_layout()
    plt.savefig(output_dir / "03_confusion_matrices.png", dpi=300)
    plt.close()

    # 4. ROC-кривые
    plt.figure(figsize=(8, 6))
    
    if len(np.unique(y_test)) > 1:
        fpr_base, tpr_base, _ = roc_curve(y_test, y_proba_base[:, 1])
        roc_auc_base = auc(fpr_base, tpr_base)
        plt.plot(fpr_base, tpr_base, label=f"Baseline LR (AUC = {roc_auc_base:.3f})", color="blue", linewidth=2)
        
        fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf[:, 1])
        roc_auc_rf = auc(fpr_rf, tpr_rf)
        plt.plot(fpr_rf, tpr_rf, label=f"Improved RF (AUC = {roc_auc_rf:.3f})", color="green", linewidth=2)
        
        plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curves Comparison")
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / "04_roc_curves.png", dpi=300)
        plt.close()

    # 5. Важность признаков
    if feature_names and len(importances) == len(feature_names):
        plt.figure(figsize=(8, 6))
        importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
        importance_df = importance_df.sort_values("importance", ascending=True).tail(10)  # Top 10
        
        plt.barh(importance_df["feature"], importance_df["importance"], color="#6c5ce7", edgecolor="black")
        plt.xlabel("Importance")
        plt.title("Feature Importances (Random Forest)")
        plt.tight_layout()
        plt.savefig(output_dir / "05_feature_importances.png", dpi=300)
        plt.close()
    
    logger.info("All plots saved")

## EDA, TRAIN, EVAL, SAVE

In [15]:
def train_and_evaluate(
    df: pd.DataFrame,
    numeric_features: List[str],
    categorical_features: List[str],
    target_column: str = "is_fit",
    test_size: float = 0.2,
    random_state: int = 42,
    metric: str = "roc_auc"
) -> tuple:
    """Обучение и оценка моделей. Возвращает результаты, y_test, preprocessor, best_model_type."""
    logger.info("Training and evaluation")
    
    numeric_features = [f for f in numeric_features if f in df.columns]
    categorical_features = [f for f in categorical_features if f in df.columns]
    
    if target_column not in df.columns:
        raise ValueError(f"Target '{target_column}' not found. Available: {df.columns.tolist()}")
    
    preprocessor = DataPreprocessor(numeric_features, categorical_features, target_column)
    X = preprocessor.fit_transform(df)
    y = df[target_column].values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    logger.info(f"Train: {len(X_train)}, Test: {len(X_test)}")
    
    results = {}
    
    # Baseline: Logistic Regression
    logger.info("Training baseline: Logistic Regression")
    baseline = LogisticRegression(max_iter=1000, random_state=random_state)
    baseline.fit(X_train, y_train)
    y_pred_base = baseline.predict(X_test)
    y_proba_base = baseline.predict_proba(X_test)
    metrics_base = ModelEvaluator.evaluate(y_test, y_pred_base, y_proba_base)
    ModelEvaluator.print_confusion_matrix(y_test, y_pred_base)
    results["baseline"] = {
        "model": "LogisticRegression", 
        "metrics": metrics_base,
        "y_pred": y_pred_base,
        "y_proba": y_proba_base
    }

    # Improved: Random Forest
    logger.info("Training improved: Random Forest")
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=random_state, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    y_proba_rf = rf.predict_proba(X_test)
    metrics_rf = ModelEvaluator.evaluate(y_test, y_pred_rf, y_proba_rf)
    ModelEvaluator.print_confusion_matrix(y_test, y_pred_rf)
    results["improved"] = {
        "model": "RandomForest", 
        "metrics": metrics_rf,
        "y_pred": y_pred_rf,
        "y_proba": y_proba_rf
    }
    
    # Выбор лучшей модели по указанной метрике
    metric = metric.lower()
    if metric not in metrics_rf or metric not in metrics_base:
        print(f"Warning: Metric '{metric}' not found. Using 'roc_auc'.")
        metric = "roc_auc"
    
    base_score = metrics_base.get(metric, 0)
    rf_score = metrics_rf.get(metric, 0)
    
    if rf_score >= base_score:
        best_model_type = "random_forest"
        best_score = rf_score
        print(f"Лучшая модель: Random Forest ({metric}={best_score:.4f})")
    else:
        best_model_type = "logistic_regression"
        best_score = base_score
        print(f"Лучшая модель: Logistic Regression ({metric}={best_score:.4f})")
    
    print("\nComparison:")
    print(f"Accuracy: {metrics_rf['accuracy'] - metrics_base['accuracy']:+.3f}")
    print(f"F1: {metrics_rf['f1'] - metrics_base['f1']:+.3f}")
    if "roc_auc" in metrics_rf and "roc_auc" in metrics_base:
        print(f"ROC-AUC: {metrics_rf['roc_auc'] - metrics_base['roc_auc']:+.3f}")
    
    return results, y_test, preprocessor, best_model_type


def save_best_model(
    df: pd.DataFrame,
    numeric_features: List[str],
    categorical_features: List[str],
    target_column: str = "is_fit",
    output_path: str = "artifacts/fitness_model.pkl",
    random_state: int = 42,
    best_model_type: str = ""
) -> tuple:
    """Сохранение лучшей модели (логистическая регрессия или случайный лес)."""
    logger.info("Saving final best model")
    
    numeric_features = [f for f in numeric_features if f in df.columns]
    categorical_features = [f for f in categorical_features if f in df.columns]
    
    pp = DataPreprocessor(numeric_features, categorical_features, target_column)
    X = pp.fit_transform(df)
    y = df[target_column].values
    
    # Создаём классификатор нужного типа
    if best_model_type == "random_forest":
        clf = FitnessClassifier(model_type="random_forest", n_estimators=100, max_depth=10, random_state=random_state)
    else:
        clf = FitnessClassifier(model_type="logistic_regression", max_iter=1000, random_state=random_state)
    
    clf.fit(X, y)
    
    output_path_obj = Path(output_path)
    if not output_path_obj.is_absolute():
        output_path_obj = project_root / output_path
    output_path_obj.parent.mkdir(parents=True, exist_ok=True)
    
    clf.save(str(output_path_obj))
    logger.info(f"Model saved to {output_path_obj}")
    
    if best_model_type == "random_forest" and hasattr(clf.model, "feature_importances_"):
        importances = clf.model.feature_importances_
        feature_names = pp.get_feature_names()
        if feature_names:
            importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
            importance_df = importance_df.sort_values("importance", ascending=False)
            print(f"Top 5 important features:\n{importance_df.head().to_string(index=False)}")
        return feature_names, importances
    else:
        feature_names = pp.get_feature_names()
        if feature_names and hasattr(clf.model, "coef_"):
            coefs = clf.model.coef_[0]
            coef_df = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
            coef_df = coef_df.reindex(coef_df["coefficient"].abs().sort_values(ascending=False).index)
            print(f"Top 5 coefficients (absolute):\n{coef_df.head().to_string(index=False)}")
        return feature_names, None

## Точка входа

In [ ]:
def main():
    """Main entry point."""
    logger.info("Starting Fitness Classifier EDA + Baseline pipeline")
    
    numeric_features = ["age", "height_cm", "weight_kg", "sleep_hours", "activity_index"]
    categorical_features = ["smokes", "gender"]
    target_column = "is_fit"
    
    KAGGLE_SLUG = "muhammedderric/fitness-classification-dataset-synthetic"
    
    try:
        df = load_dataset("data/fitness_dataset.csv", KAGGLE_SLUG)
        df = preprocess_real_dataset(df)
    except FileNotFoundError as e:
        logger.error(str(e))
        return 1
    except RuntimeError as e:
        logger.error(f"Kaggle error: {e}")
        return 1
    
    run_eda(df)
    
    results, y_test, preprocessor, best_model_type = train_and_evaluate(
        df=df,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        target_column=target_column,
        metric="roc_auc"
    )

    feature_names, importances = save_best_model(
        df=df,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        target_column=target_column,
        output_path=str(project_root / "artifacts" / "fitness_model.pkl"),
        best_model_type=best_model_type
    )
    
    artifacts_dir = project_root / "artifacts"
    save_plots(
        df=df,
        y_test=y_test,
        y_pred_base=results["baseline"]["y_pred"],
        y_pred_rf=results["improved"]["y_pred"],
        y_proba_base=results["baseline"]["y_proba"],
        y_proba_rf=results["improved"]["y_proba"],
        feature_names=feature_names if feature_names else [],
        importances=importances if importances is not None else np.array([]),
        output_dir=artifacts_dir
    )
    
    f1_improvement = results["improved"]["metrics"]["f1"] - results["baseline"]["metrics"]["f1"]
    logger.info(f"Pipeline complete. F1 improvement: {f1_improvement:+.3f}")
    logger.info(f"Plots saved to: {artifacts_dir}")
    
    return 0 if f1_improvement > 0 else 1

In [18]:
if __name__ == "__main__":
    main()

2026-05-29 23:13:04,099 | INFO     | Starting Fitness Classifier EDA + Baseline pipeline
2026-05-29 23:13:04,099 | WARNING  | File not found. Downloading from Kaggle: muhammedderric/fitness-classification-dataset-synthetic
2026-05-29 23:13:04,100 | INFO     | Loaded credentials from /Users/vladislava/Desktop/дз/ck/ai_engineering/project/configs/.env
2026-05-29 23:13:04,212 | INFO     | Downloading 'muhammedderric/fitness-classification-dataset-synthetic' to /Users/vladislava/Desktop/дз/ck/ai_engineering/project/data...


Dataset URL: https://www.kaggle.com/datasets/muhammedderric/fitness-classification-dataset-synthetic


100%|██████████████████████████████████████| 33.4k/33.4k [00:00<00:00, 1.78MB/s]
2026-05-29 23:13:05,442 | INFO     | Kaggle API download complete
2026-05-29 23:13:05,443 | INFO     | Dataset downloaded: /Users/vladislava/Desktop/дз/ck/ai_engineering/project/data/fitness_dataset.csv
2026-05-29 23:13:05,458 | INFO     | Preprocessed: 2000 rows, columns: ['age', 'height_cm', 'weight_kg', 'heart_rate', 'blood_pressure', 'sleep_hours', 'nutrition_quality', 'activity_index', 'smokes', 'gender', 'is_fit']
2026-05-29 23:13:05,459 | INFO     | EDA: Quick Checks
2026-05-29 23:13:05,478 | INFO     | Training and evaluation
2026-05-29 23:13:05,490 | INFO     | Train: 1600, Test: 400
2026-05-29 23:13:05,491 | INFO     | Training baseline: Logistic Regression
2026-05-29 23:13:05,500 | INFO     | Metrics: {'accuracy': 0.72, 'precision': 0.66, 'recall': 0.61875, 'f1': 0.6387096774193548, 'roc_auc': 0.7960416666666666}
2026-05-29 23:13:05,502 | INFO     | Confusion Matrix:
[[189  51]
 [ 61  99]]
2026-


Dataset shape: (2000, 11)
Target distribution:
is_fit
0    60.05%
1    39.95%
Name: proportion, dtype: str
First 5 rows:
   age  height_cm  weight_kg  heart_rate  blood_pressure  sleep_hours  nutrition_quality  activity_index  smokes gender  is_fit
0   56        152         65        69.6           117.0          NaN               2.37            3.97       0      F       1
1   69        186         95        60.8           114.8          7.5               8.77            3.19       0      F       1
2   46        192        103        61.4           116.4          NaN               8.20            2.03       0      F       0
3   32        189         83        60.2           130.1          7.0               6.18            3.68       0      M       1
4   60        175         99        58.1           115.8          8.0               9.95            4.83       1      F       1
Describe:
           age  height_cm  weight_kg  heart_rate  blood_pressure  sleep_hours  \
count  2000.00    2

2026-05-29 23:13:05,606 | INFO     | Metrics: {'accuracy': 0.71, 'precision': 0.6428571428571429, 'recall': 0.61875, 'f1': 0.6305732484076433, 'roc_auc': 0.785703125}
2026-05-29 23:13:05,607 | INFO     | Confusion Matrix:
[[185  55]
 [ 61  99]]
2026-05-29 23:13:05,607 | INFO     | Saving final best model
2026-05-29 23:13:05,615 | INFO     | Trained logistic_regression on 2000 samples
2026-05-29 23:13:05,619 | INFO     | Model saved to /Users/vladislava/Desktop/дз/ck/ai_engineering/project/artifacts/fitness_model.pkl
2026-05-29 23:13:05,619 | INFO     | Model saved to /Users/vladislava/Desktop/дз/ck/ai_engineering/project/artifacts/fitness_model.pkl
2026-05-29 23:13:05,621 | INFO     | Saving plots to /Users/vladislava/Desktop/дз/ck/ai_engineering/project/artifacts


Лучшая модель: Logistic Regression (roc_auc=0.7960)

Comparison:
Accuracy: -0.010
F1: -0.008
ROC-AUC: -0.010
Top 5 coefficients (absolute):
       feature  coefficient
      smokes_1    -0.984570
activity_index     0.948013
      smokes_0     0.662476
           age    -0.622385
      gender_F    -0.550350


2026-05-29 23:13:06,081 | INFO     | All plots saved
2026-05-29 23:13:06,082 | INFO     | Pipeline complete. F1 improvement: -0.008
2026-05-29 23:13:06,082 | INFO     | Plots saved to: /Users/vladislava/Desktop/дз/ck/ai_engineering/project/artifacts
